# Certior — 5-minute quickstart

**Certior is a capability boundary for AI agents.** Wrap any agent and every action is checked against a policy *before* it runs, with a signed proof attached.

This notebook runs the real `certior` package end to end. The check is **Z3 (SMT) running locally — no API key, no account, nothing to configure.** Run the cells top to bottom, then edit the inputs and re-run.

- Docs: [docs.certior.io](https://docs.certior.io) · Live demos: [playground](https://huggingface.co/spaces/paulibo/certior-playground)

In [ ]:
%pip install -q certior

## The whole idea in five lines

Declare what an agent is *allowed* to do. Then `verify()` each action: in-bounds actions are **allowed** and get a signed receipt; out-of-bounds actions are **blocked before they run**.

In [ ]:
from certior import Guard

# An agent that may read the database and make read-only HTTP calls — nothing else.
guard = Guard(policy="default", permissions=["db:read", "network:http:read"], budget_cents=1000)

# 1) An action inside the boundary -> ALLOWED, with a signed receipt.
ok = guard.verify(tool="fetch_rows", required_capabilities=["db:read"], cost_cents=5)
print("read DB   ->", "ALLOW" if ok.allowed else f"BLOCK ({ok.reason})")

# 2) An action outside the boundary -> BLOCKED before it runs.
bad = guard.verify(tool="delete_rows", required_capabilities=["db:write"], cost_cents=5)
print("write DB  ->", "ALLOW" if bad.allowed else f"BLOCK ({bad.reason})")

## The signed receipt — and the proof underneath it

Every allowed action mints a certificate. The runtime check is Z3; the *policy model* it enforces is machine-checked in **Lean 4**. The certificate carries that model's fingerprint, and the attestation tells you the one command that re-verifies it offline (0 `sorry`, standard axioms only).

In [ ]:
import json

print("certificate:")
print(json.dumps(ok.certificate.to_dict(), indent=2))

att = guard.policy_attestation
print("\npolicy model machine-checked in Lean 4")
print("fingerprint:", att["fingerprint"])
print("re-verify:  ", att["audit_command"])

## Content & budget gates

Capabilities are one of three gates. A compliance policy (here HIPAA) also scans the *content* an action would emit, and a per-agent budget caps spend. Both run in the same `verify()` call.

In [ ]:
hipaa = Guard(policy="hipaa", permissions=["email:send"], budget_cents=1000)

res = hipaa.verify(
    tool="send_email",
    required_capabilities=["email:send"],
    content="Discharge summary for John Doe, SSN 123-45-6789.",
)
print("allowed:  ", res.allowed)
print("pii_found:", res.pii_found)
print("redacted: ", res.redacted_content)

## Now make it yours

Edit any cell above and re-run:

- Add `"db:write"` to the guard's `permissions` and re-run — the blocked action turns green.
- Drop `budget_cents` below the action's `cost_cents` to see the budget gate fire.
- Change the `content` text to see what the HIPAA scanner catches.

When you're ready to wrap a real agent (OpenClaw, LangChain, CrewAI, or your own loop), the adapters are in the [docs](https://docs.certior.io/guides/custom-loop).